In [ ]:
import pandas as pd
import numpy as np
from plotnine import *
import seaborn as sns
import statsmodels.api as sm
from great_tables import GT
import requests
from io import StringIO


headers = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.8',
    'Cache-Control': 'no-cache',
    'Connection': 'keep-alive',
    'Pragma': 'no-cache',
    'Referer': 'https://download.bls.gov/pub/time.series/oe/',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'same-origin',
    'Sec-Fetch-User': '?1',
    'Sec-GPC': '1',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
    'sec-ch-ua': '"Not:A-Brand";v="99", "Brave";v="145", "Chromium";v="145"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
}

series = 'oe.data.0.Current'
response = requests.get(f'https://download.bls.gov/pub/time.series/oe/oe.{series}', headers=headers)


In [7]:

def get_bls_file(series):
    url = f'https://download.bls.gov/pub/time.series/oe/{series}'
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text), sep='\t')
        # minor cleaning
        df.columns = df.columns.str.strip()
        print("Sample data: ", df.head(n = 10))
        print("Number of Rows: ", df.shape[0])
    else:
        print("Failure to Connect: ", response.status_code)    
    return df


In [8]:
# using the other data.

df_data = get_bls_file('oe.data.0.Current')
df_series = get_bls_file("oe.series")
df_occ = get_bls_file("oe.occupation")
df_area = get_bls_file("oe.area")


Sample data:                          series_id  year period         value  footnote_codes
0  OEUM001018000000000000001       2024    A01         74090             NaN
1  OEUM001018000000000000002       2024    A01           0.0             NaN
2  OEUM001018000000000000003       2024    A01         25.74             NaN
3  OEUM001018000000000000004       2024    A01         53530             NaN
4  OEUM001018000000000000005       2024    A01           0.9             NaN
5  OEUM001018000000000000006       2024    A01         11.11             NaN
6  OEUM001018000000000000007       2024    A01         14.25             NaN
7  OEUM001018000000000000008       2024    A01         20.25             NaN
8  OEUM001018000000000000009       2024    A01         29.34             NaN
9  OEUM001018000000000000010       2024    A01         45.30             NaN
Number of Rows:  6036958


/var/folders/hr/yzpt7wbd6bjfpzfykc2jfcm80000gn/T/ipykernel_1566/1977828177.py:5: DtypeWarning: Columns (0: industry_code) have mixed types. Specify dtype option on import or set low_memory=False.


Sample data:                          series_id seasonal areatype_code industry_code  \
0  OEUM001018000000000000001             U             M             0   
1  OEUM001018000000000000002             U             M             0   
2  OEUM001018000000000000003             U             M             0   
3  OEUM001018000000000000004             U             M             0   
4  OEUM001018000000000000005             U             M             0   
5  OEUM001018000000000000006             U             M             0   
6  OEUM001018000000000000007             U             M             0   
7  OEUM001018000000000000008             U             M             0   
8  OEUM001018000000000000009             U             M             0   
9  OEUM001018000000000000010             U             M             0   

   occupation_code  datatype_code  state_code  area_code sector_code  \
0                0              1          48      10180      00--01   
1                0         

In [9]:
# merge, then pivot
df = pd.merge(df_data, df_series, on='series_id')

df_pivoted = df.pivot_table(
    index=['area_code', 'industry_code', 'occupation_code', 'year'], 
    columns='datatype_code', 
    values='value', 
    aggfunc='first'
).reset_index()

In [ ]:
df_final = pd.merge(df_pivoted, df_occ, on='occupation_code', how='left')
# Add Area Names
df_final = pd.merge(df_final, df_area, on='area_code', how='left')

# Rename columns to match the XLSX format
rename_map = {
    '01': 'TOT_EMP',
    '03': 'H_MEAN',
    '13': 'H_MEDIAN',
    'occupation_name': 'OCC_TITLE',
    'area_name': 'AREA_TITLE'
}
df_final.rename(columns=rename_map, inplace=True)

print(df_final.head())

   area_code industry_code  occupation_code  year             1             2  \
0          0             0                0  2024     154187380           0.0   
1          0             0           110000  2024      10966830           0.2   
2          0             0           111000  2024       3822780           0.3   
3          0             0           111011  2024        211850           1.2   
4          0             0           111021  2024       3584420           0.4   

              3             4             5             6  ...   16   17  \
0         32.66         67920           0.1         14.42  ...  NaN  NaN   
1         68.15        141760           0.2         27.41  ...  NaN  NaN   
2         67.24        139860           0.4         22.84  ...  NaN  NaN   
3        126.41        262930           0.9         35.44  ...  NaN  NaN   
4         64.00        133120           0.4         22.80  ...  NaN  NaN   

                         OCC_TITLE  \
0                 

In [14]:
print(df_final.shape[0])
print(df_final['year'].unique())

370980
[2024]
